# Exploring CRE → gene-tissue CLS attention in VariantFormer

This notebook demonstrates how to use the `LogAttention` callback (added in the
`feature/attention-callback` branch) to extract per-layer, per-head attention
matrices during a forward pass of VariantFormer, and how to visualize the
attention from the **gene-tissue CLS token** — a per-tissue learnable
embedding (`MultiRegistry.registry_tokens[tissue_id]`) prepended to the gene
tokens, which the model pools to predict gene expression — over the
**cis-regulatory elements (CREs)** surrounding a gene. It plays the role a
CLS token would in a vanilla transformer, but is conditioned on tissue
identity, so we call it the *gene-tissue CLS*.

By reading the **gene-tissue CLS row** (query index 0) of
the gene cross-attention matrix (`gene_modulator_crossMHA_<layer>`), we can
see which CREs the model 'looks at' when summarizing the gene for a given
tissue.

The notebook:

1. Loads a real `VF` checkpoint (`v4_ag`,
   the all-genes model).
2. Builds a single-gene single-tissue batch from the example VCF
   (`HG00096.vcf.gz`) and the reference genome.
3. Runs one forward pass under `log_attn.record_attention(...)` to capture
   the attention matrices from a few layers of the gene modulator.
4. Plots the gene-tissue CLS row of the cross-attention as a function of
   CRE genomic position, plus per-head heatmaps and a layer comparison.
5. Repeats the forward pass on the **reference genome** (no VCF) and
   compares the two attention profiles to highlight where the variants
   in `HG00096.vcf.gz` shift the model's focus.

> **Requirements**: a CUDA GPU and the artifacts produced by
> `python download_artifacts.py` (model checkpoints, reference genome, VCF).

## 1. Setup

In [ ]:
import os, sys, logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Walk up from cwd until we find the variantformer repo root (it contains
# the `processors/vcfprocessor.py` file). This works under Jupyter, VS Code,
# and `jupyter nbconvert --execute` without depending on ipynbname.
def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'processors' / 'vcfprocessor.py').exists():
            return p
    raise RuntimeError('Could not locate variantformer repo root')

REPO_PATH = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_PATH))

from processors.vcfprocessor import VCFProcessor
from seq2gene.attn_log_callback import LogAttention

logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(levelname)s %(message)s')

assert torch.cuda.is_available(), 'This notebook requires a CUDA GPU.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Repo path: {REPO_PATH}')

## 2. Pick a gene, a tissue, and a VCF

We use **APOE** (`ENSG00000130203.9`) on chromosome 19 with the example HG00096
VCF. APOE is a strongly tissue-regulated gene with well-characterized
regulatory elements, which makes it a good qualitative test case.

Feel free to change `gene_id`, `tissue`, and `vcf_path` to explore other
examples. The query format mirrors the one used in `notebooks/vcf2exp.ipynb`.

In [ ]:
MODEL_CLASS = 'v4_ag'  # all-genes checkpoint (v4_pcg is protein-coding only)
GENE_ID = 'ENSG00000130203.9'  # APOE
TISSUE = 'whole blood'
VCF_PATH = str(REPO_PATH / '_artifacts' / 'HG00096.vcf.gz')

LAYERS_TO_LOG = [0, 6, 12, 18, 24]  # Sparse selection of the 25 layers

query_df = pd.DataFrame({'gene_id': [GENE_ID], 'tissues': [TISSUE]})
query_df

## 3. Build the dataset and load the model

`VCFProcessor` handles tokenization of the gene window and the surrounding
CREs, including any variants from the VCF. We load the checkpoint manually
instead of using `Trainer.predict` so we can wrap a single forward pass in
the `record_attention` context manager.

In [ ]:
processor = VCFProcessor(model_class=MODEL_CLASS)
vcf_dataset, dataloader = processor.create_data(VCF_PATH, query_df)

model, ckpt_path, _trainer = processor.load_model()
state_dict = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model.load_state_dict(state_dict.get('state_dict', state_dict), strict=False)
_ = model.cuda().eval()

# The model reads `self.trainer.precision` to know whether to autocast.
# We use bf16-mixed (matches the training setup) and let torch.amp do the cast.
model.trainer = type('T', (), {'precision': 'bf16-mixed'})()
print('Model loaded.')

## 4. Forward pass with attention logging

`LogAttention.record_attention(model)` flips the `log_attn_matrix` flag on
every selected encoder layer of the gene modulator (and optionally the
epigenetics modulator). During the forward pass the layer recomputes the
attention matrix in PyTorch (matching FlashAttention's output up to fp16
tolerance — see `tests/test_attn_log_callback.py`) and stores it in
`layer.attn_matrix`. On context exit the callback collects, processes, and
moves the matrices to CPU and resets the flags.

In [ ]:
# keep_heads=True keeps the per-head dimension. Each per-batch entry is
# a (H, Q, K) tensor instead of being averaged to (Q, K).
log_attn = LogAttention(layer_ids=LAYERS_TO_LOG, keep_heads=True)

batch = next(iter(dataloader))
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        batch[k] = v.cuda()
    elif isinstance(v, list) and v and isinstance(v[0], torch.Tensor):
        batch[k] = [t.cuda() for t in v]

with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
    with log_attn.record_attention(model, log_epigenetics=False, log_gene=True):
        preds = model.predict_step(batch, 0)

print('Captured matrices:')
for name, mats in log_attn.attention_matrices.items():
    print(f'  {name}: list of {len(mats)} batches, first shape = {tuple(mats[0].shape)}')

pred_value = preds['pred_gene_exp'][0]
print(f'\nPredicted log-expression for {GENE_ID} in {TISSUE!r}: {float(pred_value.flatten()[0]):.3f}')

## 5. Recover CRE genomic coordinates

The dataloader doesn't pass CRE coordinates through the batch — they are
consumed only as token IDs. `VCFDataset.get_cre_positions` replays the
exact CRE-resolution pipeline used during the forward pass (manifest →
`ExtractSeqFromBed.process_subject` → strand flip), so the returned rows
align 1-to-1 with the CRE token (key) axis of the model's cross-attention
— including any rows `process_subject` drops mid-manifest when sequence
extraction fails.

In [ ]:
gene_info = vcf_dataset._get_gene_info(GENE_ID)
# Pull positions directly from the dataset so they are guaranteed to align
# with the CRE tokens the model consumed (drops anywhere in the manifest
# are honored, not just at the tail).
cre_df = vcf_dataset.get_cre_positions(GENE_ID, VCF_PATH)

cre_df['midpoint'] = (cre_df['start_cre'] + cre_df['end_cre']) // 2
gene_start_site = gene_info['start'] if gene_info['strand'] == '+' else gene_info['end']
cre_df['distance_to_gene_start'] = cre_df['midpoint'] - gene_start_site
if gene_info['strand'] == '-':
    cre_df['distance_to_gene_start'] = -cre_df['distance_to_gene_start']

print(
    f"Gene {GENE_ID} on {gene_info['chromosome']} strand={gene_info['strand']} "
    f"gene_start_site={gene_start_site}"
)
print(f'Number of CREs the model sees: {len(cre_df)}')
cre_df.head()

## 6. Inspect the captured matrices

For `Seq2GenePredictorCombinedModulator` with `gene_pooling='multi_registry'`,
the gene cross-attention matrix has shape `(num_query_tokens, num_cre_tokens)`
where `num_query_tokens = 1 (gene-tissue CLS) + num_gene_windows` and
`num_cre_tokens` equals the number of CREs surrounding the gene.

Because we passed `keep_heads=True` the callback retains the **head**
dimension, so each per-batch entry has shape
`(num_heads, num_query_tokens, num_cre_tokens)`. We average over heads here
to make the per-layer view; the per-head heatmap below uses the raw matrix
directly.

In [ ]:
# Stack per-layer attention into a dict[layer_id] -> tensor[H, Q, K]
per_head_attention = {}      # raw, per-head
gene_tissue_cls_attention = {}  # head-averaged gene-tissue CLS row vs. CRE
full_attention = {}          # head-averaged (Q, K), all gene tokens
K_attn = None                # number of CRE keys produced by the model
for name, mats in log_attn.attention_matrices.items():
    if not name.startswith('gene_modulator_crossMHA_'):
        continue
    layer_id = int(name.rsplit('_', 1)[-1])
    A_heads = mats[0].cpu().to(torch.float32).numpy()  # (H, Q, K)
    per_head_attention[layer_id] = A_heads
    A = A_heads.mean(axis=0)  # head-average: (Q, K)
    full_attention[layer_id] = A
    # The gene-tissue CLS token is the first query token.
    gene_tissue_cls_attention[layer_id] = A[0]
    K_attn = A.shape[1]

# `cre_df` already matches the model's CRE order 1-to-1 (drops included),
# and with a single-sample batch the attention K-axis is exactly the CRE
# token count — no batch-padding columns to strip. Assert this invariant
# so any future regression (e.g. running with batch_size > 1 across genes
# of differing CRE counts) fails loudly instead of silently misaligning.
assert K_attn == len(cre_df), (
    f'attention has {K_attn} keys but dataset returned {len(cre_df)} CREs; '
    f'alignment is broken.'
)
x_kb = cre_df['distance_to_gene_start'].values / 1000.0  # used by the plots below

ordered_layers = sorted(gene_tissue_cls_attention)
print('Cross-attention shapes:')
for layer_id in ordered_layers:
    print(
        f'  layer {layer_id:2d}: per_head={per_head_attention[layer_id].shape}  '
        f'full={full_attention[layer_id].shape}  '
        f'gene_tissue_cls_row={gene_tissue_cls_attention[layer_id].shape}'
    )

## 7. Plot gene-tissue CLS → CRE attention vs. distance to gene start site

Each panel shows, for one layer, how strongly the gene-tissue CLS token
attends to each surrounding CRE as a function of (signed) distance from
the gene start site. Bars increasing near 0 indicate the model relies more
on promoter-proximal CREs; bars at large distances indicate distal
enhancer-like contributions.

In [ ]:
fig, axes = plt.subplots(
    len(ordered_layers), 1, figsize=(11, 2.0 * len(ordered_layers)), sharex=True
)
if len(ordered_layers) == 1:
    axes = [axes]

x_kb = cre_df['distance_to_gene_start'].values / 1000.0
for ax, layer_id in zip(axes, ordered_layers):
    weights = gene_tissue_cls_attention[layer_id]
    ax.bar(x_kb, weights, width=2.0, color='steelblue', edgecolor='none')
    ax.axvline(0, color='red', lw=0.8, ls='--', alpha=0.7, label='gene start')
    ax.set_ylabel(f'layer {layer_id}\nattention')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[-1].set_xlabel('Distance from gene start site (kb, gene-strand orientation)')
axes[0].set_title(
    f'Gene-tissue CLS → CRE cross-attention for {GENE_ID} in {TISSUE!r}'
)
fig.tight_layout()
plt.show()

## 8. Top attended CREs (last layer)

Surfacing the actual cCRE names ranked by gene-tissue CLS attention weight is
often the
most directly interpretable view: each row corresponds to one ENCODE candidate
regulatory element.

In [ ]:
last_layer = ordered_layers[-1]
topk = 15
ranked = cre_df.copy()
ranked['attention'] = gene_tissue_cls_attention[last_layer]
ranked = ranked.sort_values('attention', ascending=False).head(topk).reset_index(drop=True)
ranked[['cre_name', 'chromosome', 'start_cre', 'end_cre', 'distance_to_gene_start', 'attention']]

## 9. Full attention heatmap (gene tokens × CREs, last layer)

Beyond the gene-tissue CLS row, the cross-attention matrix has a row per
gene window
token. Plotting the full matrix shows whether different gene windows pick
out different CREs or share roughly the same set.

In [ ]:
A = full_attention[last_layer][:, : len(cre_df)]
fig, ax = plt.subplots(figsize=(12, max(2.5, 0.4 * A.shape[0])))
im = ax.imshow(
    A,
    aspect='auto',
    cmap='magma',
    extent=[x_kb.min(), x_kb.max(), A.shape[0] - 0.5, -0.5],
    interpolation='nearest',
)
ax.axvline(0, color='cyan', lw=0.7, ls='--')
ax.set_yticks(np.arange(A.shape[0]))
y_labels = ['gene-tissue CLS'] + [f'gene window {i}' for i in range(1, A.shape[0])]
ax.set_yticklabels(y_labels)
ax.set_xlabel('Distance from gene start site (kb)')
ax.set_title(f'Cross-attention layer {last_layer} for {GENE_ID} — {TISSUE!r}')
fig.colorbar(im, ax=ax, label='attention weight')
fig.tight_layout()
plt.show()

## 10. Per-head attention at the gene-tissue CLS row (last layer)

Because we recorded with `keep_heads=True`, the head-resolved matrices are
already in `per_head_attention`. No second forward pass is needed.

In [ ]:
head_mats = per_head_attention[ordered_layers[-1]]  # (H, Q, K)
n_heads = head_mats.shape[0]
gene_tissue_cls_per_head = head_mats[:, 0, : len(cre_df)]  # (H, K_real)

fig, ax = plt.subplots(figsize=(12, max(3, 0.18 * n_heads)))
im = ax.imshow(
    gene_tissue_cls_per_head,
    aspect='auto',
    cmap='magma',
    extent=[x_kb.min(), x_kb.max(), n_heads - 0.5, -0.5],
    interpolation='nearest',
)
ax.axvline(0, color='cyan', lw=0.7, ls='--')
ax.set_xlabel('Distance from gene start site (kb)')
ax.set_ylabel('attention head')
ax.set_title(f'Per-head gene-tissue CLS → CRE attention, layer {ordered_layers[-1]}')
fig.colorbar(im, ax=ax, label='attention weight')
fig.tight_layout()
plt.show()

## 11. Repeat on the reference genome (no VCF)

Setting `vcf_path=None` makes `VCFDataset` skip `bcftools consensus` and
feed the **reference** sequence to the model. We reuse the same gene,
tissue, model, and `LayersToLog` selection, swap in a fresh
`LogAttention` callback (so the captured matrices are kept separate from
the VCF run above), and then plot the gene-tissue CLS row alongside the
VCF version at the last layer to highlight where the variants in HG00096
shift the model's attention.

In [ ]:
vcf_dataset_ref, dataloader_ref = processor.create_data(None, query_df)
log_attn_ref = LogAttention(layer_ids=LAYERS_TO_LOG, keep_heads=True)

batch_ref = next(iter(dataloader_ref))
for k, v in batch_ref.items():
    if isinstance(v, torch.Tensor):
        batch_ref[k] = v.cuda()
    elif isinstance(v, list) and v and isinstance(v[0], torch.Tensor):
        batch_ref[k] = [t.cuda() for t in v]

with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
    with log_attn_ref.record_attention(model, log_epigenetics=False, log_gene=True):
        preds_ref = model.predict_step(batch_ref, 0)

pred_ref = float(preds_ref['pred_gene_exp'][0].flatten()[0])
pred_vcf = float(pred_value.flatten()[0])
print(f'Predicted log-expression for {GENE_ID} in {TISSUE!r}:')
print(f'  reference genome : {pred_ref:.3f}')
print(f'  HG00096 VCF      : {pred_vcf:.3f}')
print(f'  delta (vcf-ref)  : {pred_vcf - pred_ref:+.3f}')

Pull the reference-genome CRE positions from the new dataset (same
alignment guarantee as in section 5) and head-average the captured
matrices, then align the K-axis exactly as we did for the VCF run.

In [ ]:
cre_df_ref = vcf_dataset_ref.get_cre_positions(GENE_ID)
cre_df_ref['midpoint'] = (cre_df_ref['start_cre'] + cre_df_ref['end_cre']) // 2
cre_df_ref['distance_to_gene_start'] = cre_df_ref['midpoint'] - gene_start_site
if gene_info['strand'] == '-':
    cre_df_ref['distance_to_gene_start'] = -cre_df_ref['distance_to_gene_start']

gene_tissue_cls_attention_ref = {}
for name, mats in log_attn_ref.attention_matrices.items():
    if not name.startswith('gene_modulator_crossMHA_'):
        continue
    layer_id = int(name.rsplit('_', 1)[-1])
    A_heads = mats[0].cpu().to(torch.float32).numpy()  # (H, Q, K)
    A = A_heads.mean(axis=0)                            # (Q, K)
    gene_tissue_cls_attention_ref[layer_id] = A[0, : len(cre_df_ref)]

x_kb_ref = cre_df_ref['distance_to_gene_start'].values / 1000.0
print(f'reference CREs: {len(cre_df_ref)} (VCF run had {len(cre_df)})')

Compare the gene-tissue CLS → CRE attention from the reference and VCF
runs at the last logged layer. Differences indicate CREs whose tokenized
sequence (and therefore the model's attention) is sensitive to the
variants carried by HG00096.

In [ ]:
last_layer = ordered_layers[-1]
attn_ref = gene_tissue_cls_attention_ref[last_layer]
attn_vcf = gene_tissue_cls_attention[last_layer]

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
for ax, (label, values, x_axis) in zip(
    axes,
    [
        ('reference genome', attn_ref, x_kb_ref),
        ('HG00096 VCF', attn_vcf, x_kb),
    ],
):
    ax.bar(x_axis, values, width=4.0, color='steelblue', alpha=0.85)
    ax.axvline(0, color='red', lw=0.8, ls='--', alpha=0.7, label='gene start')
    ax.set_ylabel(f'{label}\nattention')
    ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Distance from gene start site (kb, gene-strand orientation)')
fig.suptitle(
    f'Gene-tissue CLS → CRE attention at layer {last_layer} for {GENE_ID} — {TISSUE!r}'
)
fig.tight_layout()
plt.show()

## Recap

- `LogAttention(layer_ids=[...]).record_attention(model)` is the only thing
  needed to capture attention during a regular forward pass; it flips the
  layer-level `log_attn_matrix` flag on the way in and resets it on the way
  out.
- For `multi_registry` pooling, the **first row** of every
  `gene_modulator_crossMHA_<layer>` matrix is the gene-tissue CLS → CRE
  attention used
  for the gene-regulation analysis.
- The recompute is mathematically equivalent to FlashAttention's internal
  computation (verified in `tests/test_attn_log_callback.py::TestManualAttentionMatchesFlashAttention`).
- To go beyond a single gene, batch multiple `(gene_id, tissues)` rows in
  the query DataFrame and aggregate `log_attn.attention_matrices` across
  batches — each entry is a list of one tensor per batch.
- Passing `vcf_path=None` to `processor.create_data` runs the whole
  pipeline on the reference genome, which is useful as a baseline for
  variant-effect comparisons (section 11).